In [ ]:
import os
import pandas as pd
import yfinance as yf

os.makedirs("data", exist_ok=True)

In [ ]:
stocks = {
    "RELIANCE.NS": "Reliance Industries",
    "TCS.NS": "TCS",
    "INFY.NS": "Infosys",
    "HDFCBANK.NS": "HDFC Bank",
    "ICICIBANK.NS": "ICICI Bank"
}

In [ ]:
def fetch_news_yfinance(stocks):
    all_news = []

    for ticker, company in stocks.items():
        print(f"Fetching news for {ticker}...")

        stock = yf.Ticker(ticker)
        news = stock.news

        if not news:
            continue

        for article in news:

            title = article.get("title")
            date = article.get("providerPublishTime")
            source = article.get("publisher")


            if not title:
                content = article.get("content", {})
                title = content.get("title")
                date = content.get("pubDate")
                source = content.get("provider", {}).get("displayName")

            all_news.append({
                "date": date,
                "title": title,
                "source": source,
                "company": company,
                "ticker": ticker
            })

    df = pd.DataFrame(all_news)
    return df

In [ ]:
news_df = fetch_news_yfinance(stocks)


news_df.dropna(subset=["title"], inplace=True)

print(news_df.shape)
news_df.head()

In [ ]:
news_df.to_csv("data/news_data.csv", index=False)

In [ ]:
def fetch_multiple_stocks(stocks_dict, period="1mo"):
    all_data = []
    for ticker in stocks_dict.keys():
        print(f"Downloading data for {ticker}...")
        data = yf.download(ticker, period=period)
        if not data.empty:
            data = data.reset_index()
            data["ticker"] = ticker
            all_data.append(data)
    return pd.concat(all_data, ignore_index=True)

stock_df = fetch_multiple_stocks(stocks)

print("Stock data shape:", stock_df.shape)
stock_df.head()

In [ ]:
stock_df.to_csv("data/stock_data.csv", index=False)

In [ ]:
!pip install transformers torch --quiet


import torch
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification

In [ ]:
MODEL_NAME = "yiyanghkust/finbert-tone"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)

model.eval()

In [ ]:
news_df = pd.read_csv("data/news_data.csv")

In [ ]:
def get_sentiment_batch(text_list):
    inputs = tokenizer(
        text_list,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    with torch.no_grad():
        outputs = model(**inputs)

    scores = torch.nn.functional.softmax(outputs.logits, dim=1)


    sentiment_scores = scores[:, 2] - scores[:, 0]

    return sentiment_scores.numpy()

In [ ]:

news_sample = news_df.head(100).copy()

batch_size = 16
sentiments = []

titles = news_sample["title"].tolist()

for i in range(0, len(titles), batch_size):
    batch = titles[i:i+batch_size]
    scores = get_sentiment_batch(batch)
    sentiments.extend(scores)

news_sample["sentiment"] = sentiments

news_sample.head()

In [ ]:
batch_size = 16
sentiments = []

titles = news_df["title"].tolist()

for i in range(0, len(titles), batch_size):
    batch = titles[i:i+batch_size]
    scores = get_sentiment_batch(batch)
    sentiments.extend(scores)

news_df["sentiment"] = sentiments

In [ ]:
news_df.to_csv("data/news_with_sentiment.csv", index=False)

In [ ]:
news_df = pd.read_csv("data/news_with_sentiment.csv")
stock_df = pd.read_csv("data/stock_data.csv")

print(news_df.shape, stock_df.shape)

In [ ]:

stock_df = stock_df.reset_index(drop=True)
stock_df = stock_df.loc[:, ~stock_df.columns.duplicated()]


news_df["date"] = pd.to_datetime(news_df["date"], errors="coerce")
stock_df["Date"] = pd.to_datetime(stock_df["Date"], errors="coerce")


news_df = news_df.dropna(subset=["date", "sentiment"])

In [ ]:

news_df = news_df.sort_values("date")
stock_df = stock_df.sort_values("Date")

In [ ]:
news_df = news_df.dropna(subset=["date"])
stock_df = stock_df.dropna(subset=["Date"])

In [ ]:

news_df["date"] = news_df["date"].dt.tz_localize(None)


stock_df["Date"] = pd.to_datetime(stock_df["Date"]).dt.tz_localize(None)

In [ ]:

stock_df["Close"] = pd.to_numeric(stock_df["Close"], errors="coerce")


stock_df = stock_df.sort_values(["ticker", "Date"])


stock_df["next_close"] = stock_df.groupby("ticker")["Close"].shift(-1)


stock_df["return"] = (
    (stock_df["next_close"] - stock_df["Close"]) / stock_df["Close"]
)

In [ ]:
stock_df = stock_df.reset_index(drop=True)
news_df = news_df.reset_index(drop=True)

In [ ]:

stock_df = stock_df.sort_values("Date")
news_df = news_df.sort_values("date")

merged_df = pd.merge_asof(
    news_df,
    stock_df,
    left_on="date",
    right_on="Date",
    by="ticker",
    direction="forward"
)

In [ ]:
print(merged_df.shape)
merged_df.head()

In [ ]:
def generate_signal(sentiment):
    if sentiment > 0.3:
        return 1
    elif sentiment < -0.3:
        return -1
    else:
        return 0

merged_df["signal"] = merged_df["sentiment"].apply(generate_signal)

merged_df[["sentiment", "signal"]].head()

In [ ]:
merged_df["strategy_return"] = merged_df["signal"] * merged_df["return"]

In [ ]:
merged_df["cumulative_strategy"] = (1 + merged_df["strategy_return"]).cumprod()
merged_df["cumulative_market"] = (1 + merged_df["return"]).cumprod()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,5))

plt.plot(merged_df["cumulative_strategy"], label="Strategy")
plt.plot(merged_df["cumulative_market"], label="Market")

plt.legend()
plt.title("Strategy vs Market Performance")
plt.show()

In [ ]:
sharpe = merged_df["strategy_return"].mean() / merged_df["strategy_return"].std()
print("Sharpe Ratio:", sharpe)

In [ ]:
cum = merged_df["cumulative_strategy"]
drawdown = (cum - cum.cummax()) / cum.cummax()

print("Max Drawdown:", drawdown.min())